# 각 업종의 생산·출하 변화를 고려하면 결론이 달라질까?

재무구조가 비슷해도 업종 경기가 다르면 영업이익이 다르게 움직일 수 있다. 같은 225행에 생산증가율과 출하증가율을 하나씩 추가해 단기차입 추정치가 얼마나 변하는지 본다.

ICR(이자보상배율)은 영업이익을 이자비용으로 나눈 값이다. 3배라면 영업이익이 그해 이자비용의 세 배라는 뜻이다. 한 행은 한 업종의 한 해다. 코드의 E는 사전 단기차입 의존도, L은 사전 유동비율이다. Post는 2022~2023년이며 EP는 E와 Post를 곱한 항이다. EP의 계수는 사전 단기차입 의존도가 1%포인트 높은 업종에서 긴축기의 상대적 ICR 변화가 몇 배 달랐는지 나타낸다.


[전체 질문 목록](../README.md)

**표 읽는 법**　N은 업종·연도 관측 수, G는 업종 수다. 회귀표의 beta는 추정한 관계의 크기, lo와 hi는 95% 신뢰구간의 양 끝이다. p값은 계수가 0이라는 가정에서 지금만큼 또는 더 극단적인 검정통계량이 나올 확률이다.

# 업종 경기통제 보조분석
사용자 승인: 2026-11-03, 값이 있는 표본으로 진행. 기존 원자료/주 분석을 보존하고 새로운 보조분석으로 실행.
## 실행 전 고정한 사양
2018–2023, pre18–20 E/L 그대로 유지, Post22–23,2021 별도, 업종·연도FE. E는 단기차입+단기사채/자산의 사전 평균, L은 유동비율 사전 평균, Y=ICR 배수. 사전 노출도를 축소표본으로 재계산하지 않음.
생산·출하 원지수(2020=100)에서 g=100*ln(index_t/index_t-1). 모든 업종 지수를 동일 산식으로 처리. 생산·출하 두 g가 모두 있는 동일225행/38업종 사용. 기타전자부품·세라믹 제외, 기타수송장비2021–23만 포함. 보간·대체·극단값 삭제 없음.
F: 전체240행 기존M2 재현. C0:225행 M2. C1:C0+생산g. C2:C0+출하g. 둘 동시추가·시차·추가사양 미실행.
일반 추론=업종군집CR1, t(G−1) 신뢰구간. 동일표본 세 모형 EP/LP와 두 통제항 총8개 양측검정에 WCR-S31 Rademacher9999회 seed20261103; Holm8도 공개(이번 묶음에 한정, 전체 프로젝트 탐색 보정 아님). 핵심 해석은 EP의 방향/구간 및 동일표본 비교. 계수 변화 자체의 유의성은 별도 검정하지 않음.
경기변수는 같은 해의 경제활동이므로 금리영향의 중간경로일 수도 있음. 조건부 연관성 분석으로 해석하고 인과효과로 주장하지 않음. 메타자료에 분류 개정·집계방법 설명 없음. 코드/명칭 일치와 범위 검증을 기록하되 모집단 일치 입증이라고 하지 않음.
## 재현
이 노트북 폴더를 작업경로로 requirements.txt 환경에서 전체 셀 실행. inputs에 원 CSV, 메타TXT, 기존 동결 재무패널 및 해시 포함. 난수seed 명시. 최초 재무패널 구축은 기존01 노트북 참조.


In [1]:
# 생산·출하 자료 연결
from pathlib import Path
import pandas as pd,numpy as np,json,hashlib
BASE = Path.cwd()
(BASE / 'tables').mkdir(exist_ok=True)
for m in json.loads((BASE/'inputs/manifest.json').read_text(encoding='utf-8')):
 assert hashlib.sha256((BASE/'inputs'/m['file']).read_bytes()).hexdigest()==m['sha256']
d=pd.read_csv(BASE/'inputs/cycle.csv',header=[0,1],encoding='utf-8-sig')
b = pd.read_csv(BASE / 'inputs/base_panel.csv')
records = []
for _, r in d.iterrows():
    code, name = r.iloc[0].split(' ', 1)
    for j, c in enumerate(d.columns[1:], 1):
        year = int(c[0].split()[0][1:])
        item = c[1].split()[0]
        assert item in ['1', '2'] and '2020=100' in c[1]
        records.append(dict(code=code, industry=name, year=year, indicator={'1': 'production', '2': 'shipment'}[item], value=pd.to_numeric(r.iloc[j], errors='coerce'), raw=r.iloc[j]))
long = pd.DataFrame(records)
assert not long.duplicated(['code', 'year', 'indicator']).any()
assert set(long.year)==set(range(2017,2024))
assert (long.value.dropna()>0).all()
mapping=b[['code','industry']].drop_duplicates().merge(long[['code','industry']].drop_duplicates(),on='code',how='left',suffixes=('_panel','_cycle'),indicator=True)
assert (mapping.loc[mapping._merge=='both','industry_panel']==mapping.loc[mapping._merge=='both','industry_cycle']).all()

# 전년 대비 증가율 계산
wide=long.pivot(index=['code','industry','year'],columns='indicator',values='value').reset_index().sort_values(['code','year'])
for v in ['production','shipment']:
 wide[v+'_growth']=100*np.log(wide[v]/wide.groupby('code')[v].shift(1))
coverage=b[['code','industry','year']].merge(wide[['code','year','production_growth','shipment_growth']],on=['code','year'],how='left',validate='one_to_one')
for name,frame in [('source_long',long),('industry_mapping',mapping),('source_missing_cells',long[long.value.isna()]),('growth_series',wide),('panel_coverage',coverage),('unavailable_panel_rows',coverage[coverage.production_growth.isna()|coverage.shipment_growth.isna()])]:
 frame.to_csv(BASE/'tables'/f'{name}.csv',index=False,encoding='utf-8-sig')
print('Source rows:',len(d),'Existing industries:',b.code.nunique(),'Matching industries:',int((mapping._merge=='both').sum()))
print('Absent industries:',mapping[mapping._merge=='left_only'][['code','industry_panel']].to_dict('records'))
print('Missing source cells:',int(long.value.isna().sum()))
print('Matched available growth observations:',int(coverage[['production_growth','shipment_growth']].notna().all(axis=1).sum()),'/',len(b))
print(coverage[coverage.production_growth.isna()|coverage.shipment_growth.isna()].to_string(index=False))

# 공통 표본 확정
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from wildboottest.wildboottest import wildboottest
def save(d, name):
    d.to_csv(BASE / 'tables' / f'{name}.csv', index=False, encoding='utf-8-sig')
t=b.merge(coverage[['code','year','production_growth','shipment_growth']],on=['code','year'],validate='one_to_one')
common=t.dropna(subset=['production_growth','shipment_growth']).copy().reset_index(drop=True)
assert len(t)==240 and len(common)==225 and common.code.nunique()==38
assert len(common[common.code=='I3209'])==3
save(common, 'analysis_panel')
save(common.groupby('year').size().rename('N').reset_index(), 'sample_by_year')


Source rows: 43 Existing industries: 40 Matching industries: 38
Absent industries: [{'code': 'I3203', 'industry_panel': '기타 전자부품'}, {'code': 'I3305', 'industry_panel': '세라믹'}]
Missing source cells: 6
Matched available growth observations: 225 / 240
 code industry  year  production_growth  shipment_growth
I3203  기타 전자부품  2018                NaN              NaN
I3203  기타 전자부품  2019                NaN              NaN
I3203  기타 전자부품  2020                NaN              NaN
I3203  기타 전자부품  2021                NaN              NaN
I3203  기타 전자부품  2022                NaN              NaN
I3203  기타 전자부품  2023                NaN              NaN
I3209  기타 수송장비  2018                NaN              NaN
I3209  기타 수송장비  2019                NaN              NaN
I3209  기타 수송장비  2020                NaN              NaN
I3305      세라믹  2018                NaN              NaN
I3305      세라믹  2019                NaN              NaN
I3305      세라믹  2020                NaN              NaN
I3305     

### 출력에서 읽을 수 있는 것

재무 패널 240행 중 생산·출하 증가율을 모두 관측한 행은 225개다. 이후 같은 225행끼리 비교해야 경기지표를 넣어서 바뀐 부분을 구분할 수 있다.

- 생산·출하 지수를 연도별로 정리하고 업종코드와 이름을 대조했다.
- 증감 계산에 작년 값도 필요하므로 올해 지수가 있어도 증가율이 비어 있을 수 있다.
- 두 증가율을 모두 가진 225행을 고정했다. 다음 세 모형이 같은 업종·연도를 사용하므로 표본 변화와 경기 통제의 영향을 구분할 수 있다.

In [2]:
# 공통 추정식 정의
def design(d,terms):
    X=pd.concat([pd.DataFrame({'const':np.ones(len(d))},index=d.index),d[terms],pd.get_dummies(d.code,prefix='i',drop_first=True,dtype=float),pd.get_dummies(d.year,prefix='y',drop_first=True,dtype=float)],axis=1).astype(float)
    assert np.linalg.matrix_rank(X)==X.shape[1]
    return X
fits = {}
rows = []
def fit(label, d, terms, y='Y'):
    X = design(d, terms)
    model = sm.OLS(d[y].astype(float), X)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': d.code, 'use_correction': True, 'df_correction': True}, use_t=True)
    G = d.code.nunique()
    q = stats.t.ppf(0.975, G - 1)
    for term in terms:
        b = float(res.params[term])
        se = float(res.bse[term])
        rows.append(dict(model=label, outcome=y, term=term, beta=b, se=se, lo=b - q * se, hi=b + q * se, p=float(2 * stats.t.sf(abs(b / se), G - 1)), N=len(d), G=G))
    fits[label] = (d.copy(), terms, model, res)
    return res

terms=['EP','LP','ET','LT']

# 전체 표본 추정
fit('F_full_M2',t,terms)

# 공통 표본 추정
fit('C0_same_sample_M2',common,terms)

# 생산증가율 추가
fit('C1_production',common,terms+['production_growth'])

# 출하증가율로 교체
fit('C2_shipment',common,terms+['shipment_growth'])
coefs = pd.DataFrame(rows)
save(coefs, 'all_coefficients')
np.testing.assert_allclose(coefs.loc[(coefs.model=='F_full_M2')&(coefs.term=='EP'),'beta'],[.5459119157113599],atol=1e-10)
print(coefs.to_string(index=False))


            model outcome              term      beta       se        lo       hi        p   N  G
        F_full_M2       Y                EP  0.545912 0.451865 -0.368072 1.459896 0.234272 240 40
        F_full_M2       Y                LP  0.068685 0.066596 -0.066018 0.203388 0.308719 240 40
        F_full_M2       Y                ET -0.434758 0.323716 -1.089534 0.220019 0.187029 240 40
        F_full_M2       Y                LT  0.143002 0.113153 -0.085871 0.371875 0.213805 240 40
C0_same_sample_M2       Y                EP  0.617624 0.467445 -0.329508 1.564757 0.194521 225 38
C0_same_sample_M2       Y                LP  0.082922 0.075792 -0.070648 0.236492 0.280996 225 38
C0_same_sample_M2       Y                ET -0.362932 0.331117 -1.033839 0.307975 0.280125 225 38
C0_same_sample_M2       Y                LT  0.157069 0.122175 -0.090481 0.404619 0.206569 225 38
    C1_production       Y                EP  0.606350 0.453572 -0.312675 1.525374 0.189440 225 38
    C1_production   

### 출력에서 읽을 수 있는 것

EP의 beta를 C0→C1→C2 순서로 보면 +0.618→+0.606→+0.613이다. 세 모형의 표본이 같아 생산과 출하를 고려한 뒤 추정치가 얼마나 바뀌었는지 직접 비교할 수 있다.

- 재무구조와 긴축기 ICR 관계를 같은 방식으로 추정할 함수를 만들었다.
- 전체 240행에서 M2의 단기차입 계수는 +0.546이었다.
- 경기자료가 있는 225행만 쓰면 경기지표를 넣기 전에도 +0.618(p=0.195)로 바뀌었다. 이 차이는 표본 구성의 차이다.
- 같은 225행에 생산증가율을 추가한 계수는 +0.606(p=0.189)이었다.
- 생산 대신 출하증가율을 넣으면 +0.613(p=0.190)이었다. 같은 표본에서 두 지표 중 어느 것을 고려해도 단기차입 추정치는 비슷했다.

In [3]:
# 업종 단위 재검정

boot=[]
for label, params in [('C0_same_sample_M2', ['EP', 'LP']), ('C1_production', ['EP', 'LP', 'production_growth']), ('C2_shipment', ['EP', 'LP', 'shipment_growth'])]:
    dd, terms, model, res = fits[label]
    xx = pd.DataFrame(model.exog, columns=model.exog_names)
    gg = pd.Series(dd.code.to_numpy())
    yy = pd.Series(model.endog)
    xx = xx - xx.groupby(gg).transform('mean')
    yy = yy - yy.groupby(gg).transform('mean')
    xx = xx.loc[:, np.max(abs(xx), axis=0) > 1e-10]
    assert np.linalg.matrix_rank(xx) == xx.shape[1]
    mm = sm.OLS(yy, xx)
    for term in params:
        np.testing.assert_allclose(mm.fit().params[term], res.params[term], atol=1e-09)
        r = wildboottest(mm, param=term, cluster=pd.factorize(gg)[0], B=9999, seed=20261103, bootstrap_type='31', impose_null=True, weights_type='rademacher', show=False)
        boot.append(dict(model=label, term=term, p_boot=float(r['p-value'].iloc[0]), B=9999, seed=20261103))
z=coefs.merge(pd.DataFrame(boot),on=['model','term'],how='inner',validate='one_to_one')
assert len(z)==8
z['p_holm8'] = multipletests(z.p, method='holm')[1]
z['p_boot_holm8'] = multipletests(z.p_boot, method='holm')[1]
save(z, 'combined_inference')
print(z.to_string(index=False))


            model outcome              term     beta       se        lo       hi        p   N  G   p_boot    B     seed  p_holm8  p_boot_holm8
C0_same_sample_M2       Y                EP 0.617624 0.467445 -0.329508 1.564757 0.194521 225 38 0.207821 9999 20261103      1.0           1.0
C0_same_sample_M2       Y                LP 0.082922 0.075792 -0.070648 0.236492 0.280996 225 38 0.275528 9999 20261103      1.0           1.0
    C1_production       Y                EP 0.606350 0.453572 -0.312675 1.525374 0.189440 225 38 0.203820 9999 20261103      1.0           1.0
    C1_production       Y                LP 0.082665 0.075762 -0.070844 0.236175 0.282279 225 38 0.279728 9999 20261103      1.0           1.0
    C1_production       Y production_growth 0.042876 0.071013 -0.101010 0.186762 0.549674 225 38 0.547055 9999 20261103      1.0           1.0
      C2_shipment       Y                EP 0.612669 0.459050 -0.317455 1.542794 0.190148 225 38 0.203220 9999 20261103      1.0           1.0

### 출력에서 읽을 수 있는 것

같은 세 모형의 EP 부트스트랩 p값은 0.208, 0.204, 0.203이다. 생산·출하를 추가해도 단기차입 가설을 뒷받침하는 증거가 강해지지 않았다.

## 결과 해석

전체표본 M2: β=0.5459, 95% CI [-0.3681, 1.4599], p=0.2343. 구간에 0을 포함하므로 음·양 방향을 명확히 구별하지 못한다. 무관계 증명은 아니다.

동일표본 M2: β=0.6176, 95% CI [-0.3295, 1.5648], p=0.1945, 부트스트랩 p=0.2078. 구간에 0을 포함하므로 음·양 방향을 명확히 구별하지 못한다. 무관계 증명은 아니다.

동일표본 + 생산증가율: β=0.6063, 95% CI [-0.3127, 1.5254], p=0.1894, 부트스트랩 p=0.2038. 구간에 0을 포함하므로 음·양 방향을 명확히 구별하지 못한다. 무관계 증명은 아니다.

동일표본 + 출하증가율: β=0.6127, 95% CI [-0.3175, 1.5428], p=0.1901, 부트스트랩 p=0.2032. 구간에 0을 포함하므로 음·양 방향을 명확히 구별하지 못한다. 무관계 증명은 아니다.

동일표본 + 생산증가율의 EP 점추정 변화는 동일표본 M2 대비 -0.0113이다. 이 변화 자체의 유의성을 별도로 검정한 것은 아니다.

동일표본 + 출하증가율의 EP 점추정 변화는 동일표본 M2 대비 -0.0050이다. 이 변화 자체의 유의성을 별도로 검정한 것은 아니다.

기존 원본 보존. 전체 설명과 모든 계수는 함께 제공된 보고서 참조.